# EmoLDGN+ full-data runs on Colab or Kaggle

Runs only `b4_full` and `final_full`, for machines with 4 GB of VRAM (CLAUDE.md section 6).
Use a T4 GPU runtime. Results and checkpoints go to Drive, so a disconnected session resumes.

BRIGHTER is research-only (rule 7). Keep the Drive folder private and do not share it.

**Before the first run**, build these locally and put them in the Drive folder as one archive:

```
tar czf emoldgn_gold.tgz data/gold/tokenized data/gold/graph_bank.pt data/gold/tokenizer_meta.json
```


In [ ]:
# 1. Mount Drive and set the folder that holds emoldgn_gold.tgz and the repository archive.
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/emoldgn'
!mkdir -p $DRIVE/runs
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


In [ ]:
# 2. Install the repository. Upload it as emoldgn_repo.tgz, made locally with:
#    git archive --format=tar.gz -o emoldgn_repo.tgz HEAD
!mkdir -p /content/repo && tar xzf $DRIVE/emoldgn_repo.tgz -C /content/repo
%cd /content/repo
!pip install -q "transformers>=4.40,<5" sentencepiece accelerate pyyaml scikit-learn scipy pandas pyarrow


In [ ]:
# 3. Restore the prepared data and keep run folders on Drive.
!tar xzf $DRIVE/emoldgn_gold.tgz -C /content/repo
!rm -rf runs && ln -s $DRIVE/runs runs
!ls data/gold


In [ ]:
# 4. Train. Finished runs are skipped and unfinished ones resume, so rerun this cell after a disconnect.
!python scripts/run_experiments.py --only b4_full final_full


In [ ]:
# 5. Show progress and status.
!cat runs/b4_full/status.json runs/final_full/status.json
!tail -n 3 runs/final_full/metrics.jsonl


## Afterwards

Copy `runs/b4_full` and `runs/final_full` back to the local `runs/` folder, then finish M8 locally:

```
python scripts/run_experiments.py --evaluate
python scripts/make_figures.py --results --exp final_full
python scripts/export_artifacts.py --from-run final_full
```

Note that `freeze_word_embeddings: auto` trains the embeddings on a card with 8 GB or more,
while the local subset runs freeze them. Both choices are recorded in each run's `config.yaml`.
